# Fase 3 | Nucleo algoritmico, eficiencia y POO
Continuacion de F1/F2 con telemetria real MetroPT-3.

## Entrada: dataset limpio y trazabilidad

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.pipeline_real import *
RUTA=ROOT/'datos'/'MetroPT3(AirCompressor).csv'
print('Raiz:',ROOT)

df=cargar(RUTA)
limpios,rechazados,audit=procesar(df)
print(audit)

## Pregunta algoritmica
Comparar el recuento de observaciones con presion TP2 sobre un umbral empirico (percentil 95). Este indicador no representa una falla confirmada.

In [ ]:
import numpy as np
serie=limpios['TP2'].to_numpy(dtype=float)
umbral=float(np.quantile(serie,.95))
print('Umbral percentil 95:',umbral)

## Programacion estructurada, recursiva y vectorizada
Iterativo O(n) y espacio auxiliar O(1); recursivo divide y venceras O(n), memoria adicional por particiones/vistas y pila O(log n); vectorizado O(n) con mascara temporal O(n).

In [ ]:
muestra=serie[:min(20000,len(serie))]
resultados=[contar_iterativo(muestra,umbral),contar_recursivo(muestra,umbral),contar_vectorizado(muestra,umbral)]
print(resultados)
if len(set(resultados))!=1: raise ValueError('Resultados inconsistentes')

## Benchmark reproducible con timeit
Los tiempos dependen del equipo y del entorno. Se repiten ejecuciones con igual muestra y umbral.

In [ ]:
import timeit, pandas as pd
medidas=[]
for n in [100,1000,5000,10000]:
 a=serie[:min(n,len(serie))]
 for nombre,fn in [('iterativo',contar_iterativo),('recursivo',contar_recursivo),('vectorizado',contar_vectorizado)]:
  t=timeit.repeat(lambda:fn(a,umbral),repeat=5,number=3)
  medidas.append({'n':len(a),'algoritmo':nombre,'segundos_por_ejecucion':min(t)/3})
bench=pd.DataFrame(medidas)
display(bench)

## Interpretacion y optimizacion
Se elige la implementacion segun tiempos medidos y uso de memoria, no por suposicion. La recursividad se incluye como evidencia academica, no como recomendacion automatica.

## POO: encapsulamiento, herencia y polimorfismo
`EstrategiaResumen` define interfaz; `ResumenMedias` y `ResumenMedianas` implementan comportamientos intercambiables; `Analizador` encapsula la estrategia.

In [ ]:
a=Analizador(ResumenMedias()); print('Media TP2:',a.ejecutar(limpios)['TP2'])
a.cambiar_estrategia(ResumenMedianas()); print('Mediana TP2:',a.ejecutar(limpios)['TP2'])

## Resultados descriptivos por estado de control

In [ ]:
display(limpios.groupby('COMP')[ANALOGAS].agg(['count','mean','median']).round(3))

## Verificacion y cierre
Ejecutar pruebas con `python -m pytest -q`, guardar resultados reales y actualizar informe. No hay etiquetas de falla por fila; las diferencias observadas son descriptivas, no causales.

In [ ]:
print('Registros:',len(limpios),'Pruebas de equivalencia:',len(set(resultados))==1)